#버전 테스트 

In [1]:
import cv2
import torch
import numpy as np
import ipywidgets as widgets
from IPython.display import display

print("OpenCV version:", cv2.__version__)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("IPyWidgets version:", widgets.__version__)
print("IPython imported successfully.")

print("All basic libraries imported successfully!")

OpenCV version: 4.11.0
PyTorch version: 2.5.1+cu121
NumPy version: 2.1.2
IPyWidgets version: 8.1.7
IPython imported successfully.
All basic libraries imported successfully!


#카메라 테스트 

In [ ]:
import cv2
print(cv2.getBuildInformation())

In [4]:
import cv2

# 카메라 장치 열기 (인덱스 0: 기본 카메라)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# 카메라가 정상적으로 열렸는지 확인
if not cap.isOpened():
    print("카메라를 열 수 없습니다.")
    exit()

while True:
    # 프레임 읽기
    ret, frame = cap.read()

    # 프레임이 정상적으로 읽혔는지 확인
    if not ret:
        print("프레임을 읽을 수 없습니다.")
        break

    # 화면에 프레임 출력
    cv2.imshow('Camera', frame)

    # 'q' 키를 누르면 종료
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# 자원 해제
cap.release()
cv2.destroyAllWindows()

#가중치 불러올 수 있는지 테스트. 

In [ ]:
# 셀 1: 필요한 라이브러리 임포트 및 기본 설정
import cv2
import numpy as np
import time
import os # 파일 경로 관리를 위해 추가

# --- 설정값 (사용자 환경에 맞게 수정) ---
# 예시: 바탕화면에 Custom_YOLO4 폴더가 있다고 가정
BASE_PATH = r"C:\Users\user\Desktop\yolov4" # 실제 사용자 이름으로 변경!

# 입력 동영상 파일 경로
VIDEO_FILE_PATH = os.path.join(BASE_PATH, "test.mp4")
# YOLO 모델 파일 경로
CFG_FILE_PATH = os.path.join(BASE_PATH, "custom-test-yolo4.cfg")
WEIGHTS_FILE_PATH = os.path.join(BASE_PATH, "custom-train-yolo4_best.weights")
# 클래스 이름 파일 경로
NAMES_FILE_PATH = os.path.join(BASE_PATH, "classes.names")

# 출력 동영상 파일 이름
OUTPUT_VIDEO_NAME = "output_video_vscode.mp4"
OUTPUT_VIDEO_PATH = os.path.join(BASE_PATH, OUTPUT_VIDEO_NAME) # 출력 경로도 지정

# 탐지 관련 파라미터
MIN_CONFIDENCE = 0.3  # detection으로 인정할 최소 확률(신뢰도) 지정
NMS_THRESHOLD = 0.4   # non-max suppression threshold

# 전역 변수 초기화
writer = None
elapsed_time = 0

In [ ]:
# 셀 2: YOLO 모델 및 클래스 이름 로드
try:
    # 파일 존재 여부 확인
    if not os.path.exists(CFG_FILE_PATH):
        raise FileNotFoundError(f"CFG 파일 없음: {CFG_FILE_PATH}")
    if not os.path.exists(WEIGHTS_FILE_PATH):
        raise FileNotFoundError(f"Weights 파일 없음: {WEIGHTS_FILE_PATH}")
    if not os.path.exists(NAMES_FILE_PATH):
        raise FileNotFoundError(f"Names 파일 없음: {NAMES_FILE_PATH}")

    # Load YOLO
    net = cv2.dnn.readNetFromDarknet(CFG_FILE_PATH, WEIGHTS_FILE_PATH)
    print("YOLO 모델 로드 성공.")

    # GPU 사용 설정 (선택 사항, OpenCV가 CUDA 지원으로 빌드된 경우)
    # import torch # CUDA 사용 가능 여부 확인용
    # if torch.cuda.is_available():
    #     print("CUDA 사용 가능. DNN 백엔드를 CUDA로 설정합니다.")
    #     net.setPreferableBackend(cv2.dnn.DNN_BACKEND_CUDA)
    #     net.setPreferableTarget(cv2.dnn.DNN_TARGET_CUDA)
    # else:
    #     print("CUDA 사용 불가능. DNN 백엔드를 CPU로 설정합니다.")
    #     net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
    #     net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    classes = []
    with open(NAMES_FILE_PATH, "r") as f:
        classes = [line.strip() for line in f.readlines()]
    print(f"클래스 로드: {classes}")

    color_lists = np.random.uniform(0, 255, size=(len(classes), 3))

    model = cv2.dnn_DetectionModel(net)
    # 입력 파라미터 설정:
    # size=(416, 416)은 CFG 파일의 [net] 섹션 width, height와 일치해야 함
    # YOLOv4는 보통 416x416 또는 608x608 등을 사용
    model.setInputParams(scale=1./255, size=(416, 416), swapRB=True, crop=False)
    print("DNN Detection Model 설정 완료.")

except FileNotFoundError as e:
    print(f"오류: 파일 경로를 확인해주세요. {e}")
    # 모델 로드 실패 시 이후 코드 실행 방지
    model = None
except Exception as e:
    print(f"모델 또는 클래스 로드 중 오류 발생: {e}")
    model = None

In [ ]:
# 셀 3: 탐지 및 화면 표시 함수 정의
def detectAndDisplay(frame):
    global writer, elapsed_time, MIN_CONFIDENCE, NMS_THRESHOLD, model, classes, color_lists

    if model is None:
        print("모델이 로드되지 않아 탐지를 수행할 수 없습니다.")
        return frame # 원본 프레임 반환

    start_time = time.time()

    # classIds, scores, boxes는 튜플 형태로 반환될 수 있음
    # detect() 메서드가 classIds를 1차원 배열로 반환하는지 확인 필요 (OpenCV 버전에 따라 다를 수 있음)
    detection_results = model.detect(frame, confThreshold=MIN_CONFIDENCE, nmsThreshold=NMS_THRESHOLD)
    
    # detection_results가 (classIds, scores, boxes) 형태의 튜플인지 확인
    if isinstance(detection_results, tuple) and len(detection_results) == 3:
        classIds_flat, scores_flat, boxes = detection_results
        # classIds_flat가 2D 배열 (N,1) 형태일 수 있으므로 .flatten() 사용
        if isinstance(classIds_flat, np.ndarray) and classIds_flat.ndim > 1:
            classIds_flat = classIds_flat.flatten()
        if isinstance(scores_flat, np.ndarray) and scores_flat.ndim > 1:
            scores_flat = scores_flat.flatten()

    else:
        # 구버전 또는 다른 반환 형식일 경우 처리 (이 코드는 튜플 반환을 가정)
        print("Warning: model.detect()의 반환 형식이 예상과 다릅니다.")
        return frame


    font = cv2.FONT_HERSHEY_PLAIN
    if len(boxes) > 0 and classIds_flat is not None and scores_flat is not None: # None 체크 추가
        for i in range(len(boxes)):
            x, y, w, h = boxes[i]
            class_id = classIds_flat[i]
            score = scores_flat[i]
            
            # 클래스 ID가 유효한 범위 내에 있는지 확인
            if class_id < len(classes):
                label = '{} {:,.2%}'.format(str(classes[class_id]), score)
                # print(i, label) # 콘솔 출력은 필요시 활성화
                
                color_index = class_id % len(color_lists) # 색상 인덱스 범위 초과 방지
                color = color_lists[color_index]

                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2) # 두께를 2로 변경하여 더 잘 보이게
                # 라벨 배경을 조금 더 크게
                text_size, _ = cv2.getTextSize(label, font, 1, 1)
                cv2.rectangle(frame, (x, y - text_size[1] - 10), (x + text_size[0] + 5, y -5), color, -1)
                cv2.putText(frame, label, (x + 2, y - 10), font, 1, (255, 255, 255), 1)
            else:
                print(f"Warning: 유효하지 않은 class_id {class_id}가 감지되었습니다.")


    process_time = time.time() - start_time
    elapsed_time += process_time
    # print("=== A frame took {:.3f} seconds".format(process_time)) # 프레임별 시간은 필요시 활성화

    # 비디오를 디스크에 출력하기 위해 writer 초기화
    if writer is None and OUTPUT_VIDEO_PATH is not None:
        # 코덱 변경: "MP4V"는 .mp4 파일에 더 적합
        fourcc = cv2.VideoWriter_fourcc(*"MP4V") # 또는 "XVID"
        # 프레임 속도 (FPS)는 원본 비디오에서 가져오는 것이 좋음 (여기서는 20으로 고정)
        # 실제로는 cap.get(cv2.CAP_PROP_FPS) 사용 권장
        writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, 20,
                (frame.shape[1], frame.shape[0]), True)
        print(f"출력 비디오 파일 초기화: {OUTPUT_VIDEO_PATH}")
        
    if writer is not None:
        writer.write(frame)
    
    return frame # 처리된 프레임 반환 (실시간 표시용)

In [ ]:
# 셀 4: 비디오 처리 및 결과 표시/저장
if model is not None: # 모델이 성공적으로 로드되었을 때만 실행
    cap = cv2.VideoCapture(VIDEO_FILE_PATH)
    
    if not cap.isOpened():
        print(f'--(!)Error opening video capture: {VIDEO_FILE_PATH}')
    else:
        print(f"원본 비디오 로드 성공: {VIDEO_FILE_PATH}")
        # (선택 사항) 원본 비디오 정보 출력
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)
        print(f"원본 해상도: {width}x{height}, FPS: {fps:.2f}")

        frame_count = 0
        # VS Code에서 실시간으로 영상 띄우기 (선택적)
        # 실시간 표시는 처리 속도에 영향을 줄 수 있음
        ENABLE_REALTIME_DISPLAY = True # False로 하면 저장만 함

        while True:
            ret, frame = cap.read()
            if not ret or frame is None:
                print('--(!) No captured frame -- Break!')
                break
            
            frame_count += 1
            processed_frame = detectAndDisplay(frame) # 함수 호출

            if ENABLE_REALTIME_DISPLAY:
                # 창 크기를 조절하여 표시 (선택적)
                display_scale = 0.5 # 50% 크기로 표시
                display_width = int(processed_frame.shape[1] * display_scale)
                display_height = int(processed_frame.shape[0] * display_scale)
                cv2.imshow('YOLOv4 Detection', cv2.resize(processed_frame, (display_width, display_height)))
                
                # 'q' 키를 누르면 종료
                if cv2.waitKey(1) & 0xFF == ord('q'): # waitKey(1)로 하면 거의 실시간, 값을 늘리면 느려짐
                    break
        
        # 자원 해제
        cap.release()
        if writer is not None:
            writer.release()
            print(f"출력 비디오 저장 완료: {OUTPUT_VIDEO_PATH}")
        if ENABLE_REALTIME_DISPLAY:
            cv2.destroyAllWindows()
        
        print(f"총 처리 프레임 수: {frame_count}")
        print(f"총 경과 시간: {elapsed_time:.3f} 초")
        if frame_count > 0 :
            print(f"평균 프레임 처리 시간: {elapsed_time / frame_count:.3f} 초 (FPS: {frame_count / elapsed_time:.2f})")
else:
    print("모델이 로드되지 않아 비디오 처리를 건너뜁니다.")

#주피터 노트북 테스트 by claude 

In [6]:
import cv2
import numpy as np
import time

class YOLOv4Detector:
    def __init__(self, config_path, weights_path, classes_path):
        """
        YOLOv4 탐지기 초기화
        
        Args:
            config_path: .cfg 파일 경로
            weights_path: .weights 파일 경로  
            classes_path: .names 파일 경로
        """
        # 클래스 이름 로드
        with open(classes_path, 'r', encoding='utf-8') as f:
            self.classes = [line.strip() for line in f.readlines()]
        
        # 랜덤 색상 생성 (각 클래스별)
        self.colors = np.random.uniform(0, 255, size=(len(self.classes), 3))
        
        # YOLO 네트워크 로드
        print("YOLOv4 모델 로딩 중...")
        self.net = cv2.dnn.readNetFromDarknet(config_path, weights_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # 출력 레이어 이름 가져오기
        self.output_layers = self.net.getUnconnectedOutLayersNames()
        print(f"모델 로드 완료! 클래스 수: {len(self.classes)}")
        
    def detect(self, frame, conf_threshold=0.85, nms_threshold=0.4):
        """
        프레임에서 객체 탐지
        
        Args:
            frame: 입력 이미지
            conf_threshold: 신뢰도 임계값
            nms_threshold: NMS 임계값
            
        Returns:
            탐지 결과가 그려진 프레임
        """
        height, width = frame.shape[:2]
        
        # 이미지를 blob으로 변환
        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416, 416), swapRB=True, crop=False)
        
        # 네트워크에 입력
        self.net.setInput(blob)
        outputs = self.net.forward(self.output_layers)
        
        # 탐지 결과 파싱
        boxes = []
        confidences = []
        class_ids = []
        
        for output in outputs:
            for detection in output:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                
                if confidence > conf_threshold:
                    # 바운딩 박스 좌표 계산
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)
                    
                    # 박스의 좌상단 좌표
                    x = int(center_x - w/2)
                    y = int(center_y - h/2)
                    
                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)
        
        # Non-Maximum Suppression 적용
        indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)
        
        # 탐지된 객체들 그리기
        if len(indices) > 0:
            for i in indices.flatten():
                x, y, w, h = boxes[i]
                class_id = class_ids[i]
                confidence = confidences[i]
                
                # 바운딩 박스 그리기
                color = self.colors[class_id]
                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)
                
                # 클래스 이름과 신뢰도 표시
                label = f"{self.classes[class_id]}: {confidence:.2f}"
                label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
                
                # 레이블 배경 그리기
                cv2.rectangle(frame, (x, y - label_size[1] - 10), 
                            (x + label_size[0], y), color, -1)
                
                # 레이블 텍스트 그리기
                cv2.putText(frame, label, (x, y - 5), 
                          cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        
        return frame

def main():
    # 파일 경로 설정 (본인의 경로에 맞게 수정하세요)
    config_path = r"C:\Users\user\Desktop\yolov4\custom-test-yolo4.cfg"
    weights_path = r"C:\Users\user\Desktop\yolov4\custom-train-yolo4_best.weights"
    classes_path = r"C:\Users\user\Desktop\yolov4\classes.names"
    
    # YOLOv4 탐지기 초기화
    try:
        detector = YOLOv4Detector(config_path, weights_path, classes_path)
    except Exception as e:
        print(f"모델 로드 실패: {e}")
        return
    
    # 웹캠 초기화
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  # 0번 카메라 (기본 웹캠)
    
    if not cap.isOpened():
        print("웹캠을 열 수 없습니다!")
        return
    
    # 웹캠 해상도 설정 (선택사항)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    
    print("실시간 탐지 시작! 'q'를 눌러 종료하세요.")
    
    # FPS 계산용 변수
    fps_counter = 0
    start_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("프레임을 읽을 수 없습니다!")
            break
        
        # 객체 탐지 수행
        detected_frame = detector.detect(frame, conf_threshold=0.88, nms_threshold=0.4)
        
        # FPS 계산 및 표시
        fps_counter += 1
        elapsed_time = time.time() - start_time
        if elapsed_time >= 1.0:
            fps = fps_counter / elapsed_time
            fps_counter = 0
            start_time = time.time()
            
            # FPS 텍스트 표시
            cv2.putText(detected_frame, f"FPS: {fps:.1f}", (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # 화면에 표시
        cv2.imshow('YOLOv4 Real-time Detection', detected_frame)
        
        # 'q' 키를 누르면 종료
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # 리소스 해제
    cap.release()
    cv2.destroyAllWindows()
    print("탐지 종료!")

# 실행
if __name__ == "__main__":
    main()

YOLOv4 모델 로딩 중...
모델 로드 완료! 클래스 수: 1
실시간 탐지 시작! 'q'를 눌러 종료하세요.
탐지 종료!


# onnx 버전 by claude and gemini

In [36]:
import cv2
import numpy as np
import time

class YOLOv4Detector:
    def __init__(self, onnx_path, classes_path, input_size=416):
        """
        YOLOv4 ONNX 탐지기 초기화
        
        Args:
            onnx_path: .onnx 파일 경로
            classes_path: .names 파일 경로
            input_size: 입력 이미지 크기 (기본값: 416)
        """
        # 클래스 이름 로드
        with open(classes_path, 'r', encoding='utf-8') as f:
            self.classes = [line.strip() for line in f.readlines()]
        
        # 랜덤 색상 생성 (각 클래스별)
        self.colors = np.random.uniform(0, 255, size=(len(self.classes), 3))
        
        # 입력 크기 설정
        self.input_size = input_size
        
        # ONNX 네트워크 로드
        print("YOLOv4 ONNX 모델 로딩 중...")
        self.net = cv2.dnn.readNetFromONNX(onnx_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        
        # 출력 레이어 이름 가져오기
        self.output_layers = self.net.getUnconnectedOutLayersNames()
        print(f"ONNX 모델 로드 완료! 클래스 수: {len(self.classes)}")
        print(f"출력 레이어: {self.output_layers}")
        
    def detect(self, frame, conf_threshold=0.5, nms_threshold=0.4):
        height, width = frame.shape[:2] # 원본 프레임 크기

        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (self.input_size, self.input_size),
                                   swapRB=True, crop=False)
        self.net.setInput(blob)
        raw_outputs = self.net.forward(self.output_layers)

        scores_data = raw_outputs[0][0]
        boxes_data_norm = raw_outputs[1][0][:, 0, :] # (10647, 4) - 0-1 정규화된 값으로 가정

        boxes = []
        confidences = []
        class_ids = []

        num_detections = scores_data.shape[0]

        for i in range(num_detections):
            score = scores_data[i, 0]

            if score > conf_threshold:
                # boxes_data_norm가 (x1_norm, y1_norm, x2_norm, y2_norm) 형식이라고 가정
                # 이 값들은 0-1 범위로 정규화 되어 원본 이미지 크기에 대한 비율이라고 가정
                x1_norm, y1_norm, x2_norm, y2_norm = boxes_data_norm[i]

                x = int(x1_norm * width)
                y = int(y1_norm * height)
                # 너비와 높이 계산
                w = int((x2_norm - x1_norm) * width)
                h = int((y2_norm - y1_norm) * height)
                
                # 만약 w, h가 음수가 나오거나 매우 작다면, boxes_data_norm[i,2]와 boxes_data_norm[i,3]이
                # 너비(w_norm)와 높이(h_norm)일 수 있습니다.
                # 그 경우:
                # w_norm = boxes_data_norm[i, 2]
                # h_norm = boxes_data_norm[i, 3]
                # x = int((x1_norm - w_norm / 2) * width) # x1_norm이 cx_norm 이라면
                # y = int((y1_norm - h_norm / 2) * height) # y1_norm이 cy_norm 이라면
                # w = int(w_norm * width)
                # h = int(h_norm * height)


                boxes.append([x, y, w, h])
                confidences.append(float(score))
                class_ids.append(0)

        # print(f"임계값 통과한 탐지된 객체 수 (NMS 전): {len(boxes)}")

        if len(boxes) > 0:
            indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)

            if len(indices) > 0:
                for i_idx in indices.flatten():
                    x, y, w, h = boxes[i_idx]
                    class_id = class_ids[i_idx] 
                    confidence = confidences[i_idx]

                    color = self.colors[class_id % len(self.colors)]
                    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

                    if class_id < len(self.classes):
                        label = f"{self.classes[class_id]}: {confidence:.2f}"
                    else:
                        label = f"Class {class_id}: {confidence:.2f}" 

                    label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
                    cv2.rectangle(frame, (x, y - label_size[1] - 10),
                                (x + label_size[0], y), color, -1)
                    cv2.putText(frame, label, (x, y - 5),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)
        return frame

def main():
    # 파일 경로 설정 (본인의 경로에 맞게 수정하세요)
    onnx_path = r"C:\Users\user\Desktop\yolov4\custom-train-yolo4_best.onnx"
    classes_path = r"C:\Users\user\Desktop\yolov4\classes.names"
    
    # YOLOv4 ONNX 탐지기 초기화
    try:
        detector = YOLOv4Detector(onnx_path, classes_path, input_size=416)
    except Exception as e:
        print(f"ONNX 모델 로드 실패: {e}")
        return
    
    # 웹캠 초기화
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)  # 0번 카메라 (기본 웹캠)
    
    if not cap.isOpened():
        print("웹캠을 열 수 없습니다!")
        return
    
    # 웹캠 해상도 설정 (선택사항)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 416)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 416)
    
    print("실시간 탐지 시작! 'q'를 눌러 종료하세요.")
    
    # FPS 계산용 변수
    fps_counter = 0
    start_time = time.time()
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("프레임을 읽을 수 없습니다!")
            break
        
        # 객체 탐지 수행
        detected_frame = detector.detect(frame, conf_threshold=0.9, nms_threshold=0.4)
        
        # FPS 계산 및 표시
        fps_counter += 1
        elapsed_time = time.time() - start_time
        if elapsed_time >= 1.0:
            fps = fps_counter / elapsed_time
            fps_counter = 0
            start_time = time.time()
            
            # FPS 텍스트 표시
            cv2.putText(detected_frame, f"FPS: {fps:.1f}", (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # 화면에 표시
        cv2.imshow('YOLOv4 Real-time Detection', detected_frame)
        
        # 'q' 키를 누르면 종료
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # 리소스 해제
    cap.release()
    cv2.destroyAllWindows()
    print("탐지 종료!")

# 실행
if __name__ == "__main__":
    main()

YOLOv4 ONNX 모델 로딩 중...
ONNX 모델 로드 완료! 클래스 수: 1
출력 레이어: ('1980', 'output')
실시간 탐지 시작! 'q'를 눌러 종료하세요.
탐지 종료!


#onnx 변환 완료 후 대기 파일

In [ ]:
import cv2
import numpy as np
import time
import multiprocessing
from threading import Thread
from multiprocessing import Value
import ctypes

# 공유 변수 전역 선언
shared_center_x = multiprocessing.Value(ctypes.c_double, -1.0)
shared_center_y = multiprocessing.Value(ctypes.c_double, -1.0)
shared_confidence = multiprocessing.Value(ctypes.c_double, -1.0)

class YOLOv4Detector:
    def __init__(self, onnx_path, classes_path, input_size=416,
                 shared_center_x=None, shared_center_y=None, shared_confidence=None): # 공유 변수 추가
        """
        YOLOv4 ONNX 탐지기 초기화

        Args:
            onnx_path: .onnx 파일 경로
            classes_path: .names 파일 경로
            input_size: 입력 이미지 크기 (기본값: 416)
            shared_center_x: 가장 높은 신뢰도의 객체 x 좌표를 공유하기 위한 multiprocessing.Value
            shared_center_y: 가장 높은 신뢰도의 객체 y 좌표를 공유하기 위한 multiprocessing.Value
            shared_confidence: 가장 높은 신뢰도를 공유하기 위한 multiprocessing.Value
        """
        with open(classes_path, 'r', encoding='utf-8') as f:
            self.classes = [line.strip() for line in f.readlines()]

        self.colors = np.random.uniform(0, 255, size=(len(self.classes), 3))
        self.input_size = input_size

        print("YOLOv4 ONNX 모델 로딩 중...")
        self.net = cv2.dnn.readNetFromONNX(onnx_path)
        self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

        self.output_layers = self.net.getUnconnectedOutLayersNames()
        print(f"ONNX 모델 로드 완료! 클래스 수: {len(self.classes)}")
        print(f"출력 레이어: {self.output_layers}")

        # 공유 변수 저장
        self.shared_center_x = shared_center_x
        self.shared_center_y = shared_center_y
        self.shared_confidence = shared_confidence

    def detect_and_get_best_target(self, frame, conf_threshold=0.5, nms_threshold=0.4):
        """
        프레임에서 객체 탐지, 가장 신뢰도 높은 객체 정보 반환 및 공유 변수 업데이트

        Args:
            frame: 입력 이미지
            conf_threshold: 신뢰도 임계값
            nms_threshold: NMS 임계값

        Returns:
            (탐지 결과가 그려진 프레임, (가장 높은 신뢰도의 객체 중앙 x, 가장 높은 신뢰도의 객체 중앙 y, 신뢰도))
            또는 객체가 없으면 (탐지 결과가 그려진 프레임, None)
        """
        original_height, original_width = frame.shape[:2]

        blob = cv2.dnn.blobFromImage(frame, 1/255.0, (self.input_size, self.input_size),
                                   swapRB=True, crop=False)
        self.net.setInput(blob)
        raw_outputs = self.net.forward(self.output_layers)

        # 가설 3 기반 파싱
        # raw_outputs[0]가 점수 (1, N, 1), raw_outputs[1]가 박스 (1, N, 1, 4) 또는 그 반대
        # 출력 순서 확인 필요: 현재는 scores_data가 첫 번째, boxes_data_norm이 두 번째로 가정
        # self.output_layers 순서와 raw_outputs shape으로 재확인 가능
        # '1980' (1, 10647, 1) -> scores, 'output' (1, 10647, 1, 4) -> boxes
        if raw_outputs[0].shape[-1] == 1 and raw_outputs[1].shape[-1] == 4: # 점수, 박스 순서
            scores_data = raw_outputs[0][0]           # (N, 1)
            boxes_data_norm = raw_outputs[1][0][:,0,:] # (N, 4)
        elif raw_outputs[1].shape[-1] == 1 and raw_outputs[0].shape[-1] == 4: # 박스, 점수 순서
            scores_data = raw_outputs[1][0]           # (N, 1)
            boxes_data_norm = raw_outputs[0][0][:,0,:] # (N, 4)
        else:
            print("모델 출력 순서나 형태를 다시 확인해야 합니다.")
            if self.shared_confidence is not None: self.shared_confidence.value = -1.0 # 탐지 실패
            if self.shared_center_x is not None: self.shared_center_x.value = -1.0
            if self.shared_center_y is not None: self.shared_center_y.value = -1.0
            return frame, None


        boxes = []
        confidences = []
        class_ids = [] # 단일 클래스 가정

        num_detections = scores_data.shape[0]

        for i in range(num_detections):
            score = scores_data[i, 0]

            if score > conf_threshold:
                # 가설 3: boxes_data_norm는 (x1_norm, y1_norm, x2_norm, y2_norm)
                # 0-1 정규화, 원본 이미지 크기 기준
                x1_norm, y1_norm, x2_norm, y2_norm = boxes_data_norm[i]

                x = int(x1_norm * original_width)
                y = int(y1_norm * original_height)
                w = int((x2_norm - x1_norm) * original_width)
                h = int((y2_norm - y1_norm) * original_height)

                # 유효한 박스인지 간단히 확인 (w, h > 0)
                if w > 0 and h > 0:
                    boxes.append([x, y, w, h])
                    confidences.append(float(score))
                    class_ids.append(0) # 단일 클래스

        best_target_info = None
        highest_confidence = -1.0
        best_box_center_x = -1.0
        best_box_center_y = -1.0

        if len(boxes) > 0:
            indices = cv2.dnn.NMSBoxes(boxes, confidences, conf_threshold, nms_threshold)

            if len(indices) > 0:
                # 가장 높은 신뢰도를 가진 객체 찾기
                for i_idx_flat in indices.flatten(): # indices가 (N,1) 형태일 수 있으므로 flatten()
                    current_confidence = confidences[i_idx_flat]
                    if current_confidence > highest_confidence:
                        highest_confidence = current_confidence
                        x_b, y_b, w_b, h_b = boxes[i_idx_flat]
                        best_box_center_x = x_b + w_b / 2
                        best_box_center_y = y_b + h_b / 2

                # 모든 탐지된 객체 그리기 (이전과 동일)
                for i_idx_flat in indices.flatten():
                    x, y, w, h = boxes[i_idx_flat]
                    class_id = class_ids[i_idx_flat]
                    confidence = confidences[i_idx_flat]

                    color = self.colors[class_id % len(self.colors)]
                    cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

                    label = f"{self.classes[class_id]}: {confidence:.2f}"
                    label_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)[0]
                    cv2.rectangle(frame, (x, y - label_size[1] - 10),
                                (x + label_size[0], y), color, -1)
                    cv2.putText(frame, label, (x, y - 5),
                              cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

                if highest_confidence > -1.0 : # 가장 신뢰도 높은 객체가 있다면
                    best_target_info = (best_box_center_x, best_box_center_y, highest_confidence)
                    # 가장 신뢰도 높은 객체에 특별한 표시 (선택 사항)
                    cv2.circle(frame, (int(best_box_center_x), int(best_box_center_y)), 10, (0,0,255), -1)
                    cv2.putText(frame, "BEST", (int(best_box_center_x) + 15, int(best_box_center_y) + 5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)


        # 공유 변수 업데이트
        if self.shared_center_x is not None:
            self.shared_center_x.value = float(best_box_center_x) # multiprocessing.Value는 float/int 등 기본타입 직접 할당
        if self.shared_center_y is not None:
            self.shared_center_y.value = float(best_box_center_y)
        if self.shared_confidence is not None:
            self.shared_confidence.value = float(highest_confidence)

        return frame, best_target_info


def main():
    # 파일 경로 설정
    onnx_path = r"C:\Users\user\Desktop\yolov4\custom-train-yolo4_best.onnx" # 실제 경로로 수정
    classes_path = r"C:\Users\user\Desktop\yolov4\classes.names"        # 실제 경로로 수정

    # multiprocessing.Value 생성 (d: double-precision float, i: integer)
    # main 프로세스에서 생성하고 Detector에 전달
    shared_target_x = multiprocessing.Value('d', -1.0)
    shared_target_y = multiprocessing.Value('d', -1.0)
    shared_target_conf = multiprocessing.Value('d', -1.0)


    try:
        
        detector = YOLOv4Detector(
                        onnx_path=r"C:\Users\user\Desktop\yolov4\custom-train-yolo4_best.onnx",
                        classes_path=r"C:\Users\user\Desktop\yolov4\classes.names",
                        input_size=416,
                        shared_center_x=shared_center_x,
                        shared_center_y=shared_center_y,
                        shared_confidence=shared_confidence
        )
    except Exception as e:
        print(f"ONNX 모델 로드 실패: {e}")
        return

    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
    if not cap.isOpened():
        print("웹캠을 열 수 없습니다!")
        return

    # 웹캠 해상도 설정 시도 (실제 적용 여부는 웹캠에 따라 다름)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 416)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 416)

    print("실시간 탐지 시작! 'q'를 눌러 종료하세요.")
    print("외부에서 shared_target_x.value, shared_target_y.value, shared_target_conf.value 로 접근 가능합니다.")

    fps_counter = 0
    start_time = time.time()
    accumulated_fps_values = [] # FPS 값들을 저장할 리스트

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("프레임을 읽을 수 없습니다!")
                break

            # 객체 탐지 및 가장 좋은 타겟 정보 가져오기
            processed_frame, best_target = detector.detect_and_get_best_target(frame, conf_threshold=0.9, nms_threshold=0.4)

            # FPS 계산 및 표시
            fps_counter += 1
            elapsed_time = time.time() - start_time
            if elapsed_time >= 1.0:
                fps = fps_counter / elapsed_time
                accumulated_fps_values.append(fps) # FPS 값 저장
                fps_counter = 0
                start_time = time.time()

                # FPS 텍스트 표시 (가장 최근 FPS)
                cv2.putText(processed_frame, f"FPS: {fps:.1f}", (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

            # 현재 공유되고 있는 좌표값 화면에 표시 (디버깅용)
            # cv2.putText(processed_frame, f"Shared X: {shared_target_x.value:.1f}", (10, 60),
            #            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            # cv2.putText(processed_frame, f"Shared Y: {shared_target_y.value:.1f}", (10, 90),
            #            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)
            # cv2.putText(processed_frame, f"Shared Conf: {shared_target_conf.value:.2f}", (10, 120),
            #            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)


            cv2.imshow('YOLOv4 Real-time Detection', processed_frame)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    finally:
        # 루프 종료 후 평균 FPS 계산 및 출력
        if accumulated_fps_values:
            avg_fps = sum(accumulated_fps_values) / len(accumulated_fps_values)
            print(f"\n평균 FPS: {avg_fps:.2f}")
        else:
            print("\nFPS 측정 데이터가 없습니다.")

        cap.release()
        cv2.destroyAllWindows()
        print("탐지 종료!")
        # 최종 공유 값 확인
        print(f"최종 공유된 X 좌표: {shared_target_x.value}")
        print(f"최종 공유된 Y 좌표: {shared_target_y.value}")
        print(f"최종 공유된 신뢰도: {shared_target_conf.value}")

if __name__ == "__main__":
    # Jupyter Notebook 등에서 multiprocessing 사용 시 필요할 수 있는 설정
    # multiprocessing.freeze_support() # Windows에서 .exe로 빌드 시 필요
    # multiprocessing.set_start_method('spawn', force=True) # 경우에 따라 필요

    main()

    # main 함수 실행 후, 다른 셀이나 스레드에서 아래와 같이 값에 접근 가능 (main이 실행 중이거나, Detector 객체가 살아있어야 함)
    # 이 예제에서는 main 함수가 끝나면 프로그램이 종료되므로,
    # 실제 다른 프로세스에서 사용하려면 main 루프를 백그라운드 스레드로 돌리거나,
    # Detector를 사용하는 별도의 프로세스를 만들어야 합니다.
    #
    # 여기서는 main()이 끝난 후 값을 출력해보는 것은 의미가 크지 않지만,
    # 개념적으로는 이렇게 접근한다는 것을 보여줍니다.
    # print(f"Main 함수 종료 후 접근: X={shared_target_x.value}, Y={shared_target_y.value}, Conf={shared_target_conf.value}")
    # 위 주석은 main 함수 스코프 밖에 shared_target_x 등이 선언되어야 의미가 있습니다.
    # 지금 구조에서는 main 함수 내에서 선언하고 사용 후 종료됩니다.
    #
    # 만약 Jupyter Notebook에서 이 셀을 실행하고, 다음 셀에서 값을 확인하고 싶다면,
    # shared_target_x, y, conf를 main() 함수 바깥, 즉 전역 스코프에 선언해야 합니다.
    # 그리고 main() 함수는 이 전역 변수를 사용하도록 수정해야 합니다.
    # (또는 Detector를 전역으로 만들고, main은 단순히 루프만 실행)
    # 스레드로 main() 실행


YOLOv4 ONNX 모델 로딩 중...
ONNX 모델 로드 완료! 클래스 수: 1
출력 레이어: ('1980', 'output')
실시간 탐지 시작! 'q'를 눌러 종료하세요.
외부에서 shared_target_x.value, shared_target_y.value, shared_target_conf.value 로 접근 가능합니다.

평균 FPS: 2.17
탐지 종료!
최종 공유된 X 좌표: -1.0
최종 공유된 Y 좌표: -1.0
최종 공유된 신뢰도: -1.0


#실행 셀 

In [ ]:
from threading import Thread

# 별도의 스레드에서 main() 실행
thread = Thread(target=main, daemon=True)
thread.start()

YOLOv4 ONNX 모델 로딩 중...


ONNX 모델 로드 완료! 클래스 수: 1
출력 레이어: ('1980', 'output')
실시간 탐지 시작! 'q'를 눌러 종료하세요.
외부에서 shared_target_x.value, shared_target_y.value, shared_target_conf.value 로 접근 가능합니다.

평균 FPS: 2.98
탐지 종료!
최종 공유된 X 좌표: -1.0
최종 공유된 Y 좌표: -1.0
최종 공유된 신뢰도: -1.0


#전역 변수 송신 셀 

In [70]:
import time

try:
    while True:
        x = shared_center_x.value
        y = shared_center_y.value
        conf = shared_confidence.value
        
        print(f"\r좌표: ({x:.1f}, {y:.1f}), 신뢰도: {conf:.2f}", end="")
        
        time.sleep(0.1)  # 너무 빠르면 CPU 낭비되니까 약간 쉬었다가 다시 읽기
except KeyboardInterrupt:
    print("\n\n실시간 모니터링 종료")

좌표: (-1.0, -1.0), 신뢰도: -1.001

실시간 모니터링 종료


# 그냥 달려! 

In [ ]:
import time
from pop import Pilot

autocar = Pilot.AutoCar()

# 고정 속도 20%
SPEED = 20

# x좌표 범위 설정
X_MIN = 100
X_MAX = 540
X_CENTER = 320

try:
    while True:
        # shared_center_x.value라고 가정한 변수 (시뮬레이션용)
        x = shared_center_x.value  # 이 부분은 실제 환경에 맞게 대체

        if x == -1 or x < X_MIN or x > X_MAX:
            steer = 0  # 인식 실패 시 직진
        else:
            # 선형 보간 계산
            steer = (x - X_CENTER) / 220
            # steering은 -1 ~ 1 사이여야 함
            steer = max(-1.0, min(1.0, steer))  # 안전하게 클램프

        # 조향 및 속도 적용
        autocar.steering = steer
        autocar.forward(SPEED)

        print(f"\r조향값: {steer:.3f}, x좌표: {x:.1f}", end="")

        time.sleep(0.1)  # 너무 빠르면 CPU 낭비되므로 약간 쉬었다가 다시 읽기

except KeyboardInterrupt:
    print("\n\n자동 주행 종료")
    autocar.stop()